# Chunked English→German translation with resumable merging

This notebook splits `english_news_v13.csv` into 10 smaller batches, translates only the `title` and `description` fields of each batch into German via `deep-translator`, and after every batch appends the result to `merged_news_v1.csv`. Each translated chunk is stored on disk with a progress log so rerunning the notebook skips completed batches automatically.


In [7]:
from __future__ import annotations

import json
from pathlib import Path
from typing import List, Set

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

DATA_DIR = Path("/Users/nicolas/Desktop/Repos/zhaw_arep/data_prep")
EN_INPUT = DATA_DIR / "english_news_v13.csv"
GERMAN_INPUT = DATA_DIR / "german_news_v3.csv"
CHUNK_OUTPUT_DIR = DATA_DIR / "translated_chunks_v13"
MERGED_OUTPUT = DATA_DIR / "merged_news_v1.csv"
PROGRESS_FILE = DATA_DIR / "translation_progress.json"
CHUNK_COUNT = 10
COLUMNS_TO_TRANSLATE = ["title", "description"]
TARGET_LANG = "de"
SOURCE_LANG = "en"


def ensure_directories() -> None:
    CHUNK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def chunk_output_path(chunk_idx: int) -> Path:
    return CHUNK_OUTPUT_DIR / f"english_news_v13_chunk_{chunk_idx:02d}_translated.csv"


def chunk_label(chunk_idx: int) -> str:
    return f"chunk_{chunk_idx:02d}"


def load_progress() -> Set[int]:
    if PROGRESS_FILE.exists():
        data = json.loads(PROGRESS_FILE.read_text())
        return set(data.get("completed_chunks", []))
    return set()


def save_progress(completed_chunks: Set[int]) -> None:
    payload = {"completed_chunks": sorted(completed_chunks)}
    PROGRESS_FILE.write_text(json.dumps(payload, indent=2))


def split_dataframe(df: pd.DataFrame, parts: int) -> List[pd.DataFrame]:
    return [chunk.reset_index(drop=True) for chunk in np.array_split(df, parts)]


def translate_text(text: str, translator) -> str:
    if not isinstance(text, str) or not text.strip():
        return text
    try:
        return translator.translate(text)
    except Exception as exc:
        print(f"Translation failed for '{text[:30]}...': {exc}")
        return text


def translate_chunk(chunk_df: pd.DataFrame, translator, chunk_idx: int) -> pd.DataFrame:
    work_df = chunk_df.copy()
    work_df["chunk_id"] = chunk_idx
    for column in COLUMNS_TO_TRANSLATE:
        if column in work_df.columns:
            tqdm.pandas(desc=f"Chunk {chunk_idx:02d} · {column}")
            # Replace the original column with German translation (not create _de column)
            work_df[column] = work_df[column].progress_apply(
                lambda text: translate_text(text, translator)
            )
    return work_df


def ensure_merged_base() -> None:
    if MERGED_OUTPUT.exists():
        return
    german_df = pd.read_csv(GERMAN_INPUT)
    german_df.to_csv(MERGED_OUTPUT, index=False)


def append_chunk_to_merged(chunk_df: pd.DataFrame) -> None:
    header_needed = not MERGED_OUTPUT.exists()
    chunk_df.to_csv(MERGED_OUTPUT, mode="a", header=header_needed, index=False)



In [8]:
from deep_translator import GoogleTranslator

ensure_directories()
ensure_merged_base()

english_df = pd.read_csv(EN_INPUT)
if english_df.empty:
    raise ValueError("English dataset is empty—nothing to translate.")

chunks = split_dataframe(english_df, CHUNK_COUNT)
translator = GoogleTranslator(source=SOURCE_LANG, target=TARGET_LANG)
completed_chunks = load_progress()
processed_chunks: list[int] = []
skipped_chunks: list[int] = []
reused_translations: list[int] = []

for idx, chunk_df in enumerate(chunks, start=1):
    label = chunk_label(idx)
    chunk_path = chunk_output_path(idx)

    if idx in completed_chunks:
        skipped_chunks.append(idx)
        print(f"Skipping {label}: already merged.")
        continue

    if chunk_df.empty and not chunk_path.exists():
        print(f"{label} contains no rows—marking complete.")
        completed_chunks.add(idx)
        save_progress(completed_chunks)
        skipped_chunks.append(idx)
        continue

    if chunk_path.exists():
        print(f"Reusing existing translation for {label}.")
        translated_chunk = pd.read_csv(chunk_path)
        reused_translations.append(idx)
    else:
        translated_chunk = translate_chunk(chunk_df, translator, idx)
        translated_chunk.to_csv(chunk_path, index=False)
        print(f"Saved translated {label} to {chunk_path.name}.")

    append_chunk_to_merged(translated_chunk)
    completed_chunks.add(idx)
    save_progress(completed_chunks)
    processed_chunks.append(idx)
    print(f"Merged {label} into {MERGED_OUTPUT.name}.")

run_summary = {
    "processed_chunks": processed_chunks,
    "skipped_chunks": skipped_chunks,
    "reused_translations": reused_translations,
    "progress_file": PROGRESS_FILE,
    "merged_output": MERGED_OUTPUT,
    "chunk_dir": CHUNK_OUTPUT_DIR,
}
print("Translation pipeline complete.")


Skipping chunk_01: already merged.
Skipping chunk_02: already merged.
Skipping chunk_03: already merged.
Skipping chunk_04: already merged.
Skipping chunk_05: already merged.
Skipping chunk_06: already merged.
Skipping chunk_07: already merged.
Skipping chunk_08: already merged.
Skipping chunk_09: already merged.
Skipping chunk_10: already merged.
Translation pipeline complete.


/Users/nicolas/Desktop/Repos/zhaw_mle/.conda/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [9]:
if "run_summary" not in globals():
    raise RuntimeError("Run the translation cell first to populate the summary data.")

print("Processed chunks this run:", run_summary["processed_chunks"])
print("Skipped chunks this run:", run_summary["skipped_chunks"])
print("Reused translations this run:", run_summary["reused_translations"])
print("Progress log:", run_summary["progress_file"])
print("Chunk directory:", run_summary["chunk_dir"])
print("Merged output:", run_summary["merged_output"])

if run_summary["merged_output"].exists():
    try:
        merged_preview = pd.read_csv(run_summary["merged_output"], on_bad_lines='skip').tail(5)
        display(merged_preview)
    except Exception as e:
        print(f"Failed to read merged output: {e}")
else:
    print("Merged output file does not exist yet.")


Processed chunks this run: []
Skipped chunks this run: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Reused translations this run: []
Progress log: /Users/nicolas/Desktop/Repos/zhaw_arep/data_prep/translation_progress.json
Chunk directory: /Users/nicolas/Desktop/Repos/zhaw_arep/data_prep/translated_chunks_v13
Merged output: /Users/nicolas/Desktop/Repos/zhaw_arep/data_prep/merged_news_v1.csv


,publishedAt,title,source,description,url
113889,2025-10-31 14:31:34+00:00,China-Spionage? - USA wollen Router von TP Lin...,Bild,TP-Link steht vor Verkaufsverbot in den USA we...,https://www.bild.de/leben-wissen/digital/china...
113890,2025-10-31 15:27:48+00:00,Kiew zerstört in Geheim-Operation eine von nur...,Focus,Ukraine-News im Live-Ticker: Alle aktuellen En...,https://www.focus.de/politik/selenskyj-ukraine...
113891,2025-10-31 15:58:19+00:00,Europäischer Emissionshandel: So sabotiert man...,Die Zeit,In der EU wächst der Widerstand gegen das Herz...,https://www.zeit.de/2025/46/europaeischer-emis...
113892,2025-10-31 16:24:42+00:00,"Ukraine-Invasion, Tag 1345: Wie Selenskyj Trum...",Der Tagesspiegel,Russland soll verbotenen Marschflugkörper eing...,https://www.tagesspiegel.de/internationales/uk...
113893,2025-10-31 16:32:16+00:00,Interne Kurse enthüllt - So sollen Reporter vo...,Bild,BILD liegen Schulungsunterlagen des gemeinsame...,https://www.bild.de/politik/inland/interne-kur...
